# colab39 — how many random tie orderings does MAP@10 need?

**Question.** MAP@10 is reported as the mean over **200 independent random orderings within tied
groups**. Where 200 came from is not recorded. This notebook measures whether it matters.

**What it produces.** For every method/dataset combination that actually has ties:

* the spread of MAP@10 across single random orderings (the standard deviation $\sigma$),
* the standard error of the reported estimate at $B = 200$, which is $\sigma/\sqrt{B}$,
* a convergence curve of the running estimate against $B$, from 1 to 1,000.

If the curve is flat well before 200 and the standard error at 200 is far below the precision the
thesis reports, then the choice of 200 provably did not matter — which is the whole defence.

**Scope.** Only the two methods that produce ties systematically:

* the **length-ratio baseline** — every pair with the same two lengths gets the same score,
* the **Dice coefficient over 3-grams** — on SS especially, where a 3-symbol alphabet admits at
  most 27 distinct 3-grams, so the coefficient is a ratio of small integers.

Both are computed from the strings alone. **No encoder checkpoint, no ESM-2, no GPU.**

**This notebook changes no protocol and adds no baseline.** It measures the existing one.

⚠ Runtime is dominated by the exact relevance-set scan (one full pairwise pass per collection,
the same scan colab37 runs). SS is the slow one — expect tens of minutes. Everything after it is
fast.

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet

## 1. Constants — identical to the run of record

Copied verbatim from colab37. Do not change any of these without changing the thesis.

In [ ]:
import numpy as np, pandas as pd, json, time
import matplotlib.pyplot as plt
from scipy import sparse
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist

DATA_DIR = 'sampledata/cath'

# --- IDENTICAL to the run of record. Do not change without changing the thesis. ---
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
MIN_LEN, MAX_LEN = 50, 200
RESCUED   = {'4z0mC02', '3qkaE02'}   # kept per the 2026-08-24 decision; disclosed in Section 3.4
HIGH      = 0.70                     # the high range, Table 3.1
K_AT      = 10                       # MAP@10
SYN_PERTURB, SYN_INDEP = 20_000, 8_000
SYN_SEED, STRAT_PER_BIN = 20260810, 400

# --- this notebook's own knobs ---
B_MAX      = 1_000                   # orderings drawn per query
B_REPORTED = 200                     # the constant under examination
TIE_SEED   = 20260825

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

# LOCKED DECK PALETTE (confirmed 2026-08-24): synth orange / 3Di blue / SS red / AA dark grey.
COLOUR = {'Synth': '#FF7F0E', '3Di': '#0072B2', 'SS': '#D62728', 'AA': '#4D4D4D'}
ORDER  = ['Synth', 'AA', '3Di', 'SS']
print('constants loaded')

## 2. Collections

Same filter as the run of record, with the same asserts. If any of these fail, **stop** — the rest
of the notebook would describe a different dataset than the thesis.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}

COLLECTION = {'AA': list(id_to_aa.values()),
              'SS': list(id_to_ss.values()),
              '3Di': list(id_to_3di.values())}

for r in ['AA', 'SS', '3Di']:
    print(f'  {r:<4} sequences = {len(COLLECTION[r]):>6,}')
assert len(COLLECTION['AA'])  == 10_501, 'AA collection differs from the run of record - STOP'
assert len(COLLECTION['SS'])  == 10_497, 'SS collection differs from the run of record - STOP'
assert len(COLLECTION['3Di']) == 10_501, '3Di collection differs from the run of record - STOP'
print('matches the run of record')

In [ ]:
# --- the synthetic EVALUATION collection, verbatim from the run of record ---
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1))
    return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:            op = 'ins'
        elif len(s) >= MAX_LEN:    op = rng.choice(['sub', 'del'])
        else:                      op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=SYN_SEED):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base) + 1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN:
            recs.append((base, part))
    for _ in range(n_indep):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size:
            take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs = []
    for idx in take:
        a, b, l = recs[int(idx)]
        seqs.append(a); seqs.append(b)
    return seqs

print('generating the Synth evaluation collection - about a minute...')
COLLECTION['Synth'] = build_synth_feed(SYN_PERTURB, SYN_INDEP)
print(f'  Synth Eval collection = {len(COLLECTION["Synth"]):,} sequences (run of record: 7,296)')
assert len(COLLECTION['Synth']) == 7_296, 'Synth Eval collection differs from the run of record - STOP'
print('matches the run of record')

## 3. Exact relevance sets

Every sequence against every other sequence in the same collection, exact normalised Levenshtein.
A neighbour is relevant at $s_{\mathrm{Lev}} \ge 0.70$. Only membership is kept, not the matrix.

⚠ **This is the expensive cell.** One full pairwise pass per collection. SS takes tens of minutes.

The asserts at the end are the check that this reproduces the protocol: the high-similarity pair
counts should come out at **AA 5 · 3Di 6,009 · SS 623,077**, and the resulting query counts should
match the evaluation-set sizes recorded in Section 3.6.7.

In [ ]:
def relevance_sets(seqs, block=512, tag=''):
    '''Returns list of arrays: for each sequence, the indices of its relevant neighbours.'''
    N = len(seqs); lens = np.array([len(s) for s in seqs])
    rel = [[] for _ in range(N)]
    n_pairs_high = 0
    t0 = time.time()
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - D / den
        for i in range(r1 - r0):
            gi = r0 + i
            hit = np.where(sim[i] >= HIGH)[0]
            hit = hit[hit != gi]                      # never its own neighbour
            rel[gi] = hit.astype(np.int32)
            n_pairs_high += int((hit > gi).sum())     # count each unordered pair once
        print(f'    {tag} rows {r1:>6,} / {N:,}   ({time.time()-t0:.0f}s)', end='\r')
    print()
    return rel, n_pairs_high

REL, HIGH_PAIRS = {}, {}
for r in ORDER:
    print(f'{r}: exact relevance-set scan over {len(COLLECTION[r]):,} sequences')
    REL[r], HIGH_PAIRS[r] = relevance_sets(COLLECTION[r], tag=r)
    nq = sum(1 for x in REL[r] if len(x) > 0)
    print(f'  {r:<6} high-similarity pairs = {HIGH_PAIRS[r]:>9,}   eligible queries = {nq:>7,}')

print('\nrecorded values --  pairs: AA 5 / 3Di 6,009 / SS 623,077')
print('             queries: AA 10 / 3Di 347 / SS 10,002 / Synth 2,410  (Section 3.6.7)')
assert HIGH_PAIRS['AA'] == 5,        'AA high-similarity pair count differs - STOP'
assert HIGH_PAIRS['3Di'] == 6_009,   '3Di high-similarity pair count differs - STOP'
assert HIGH_PAIRS['SS'] == 623_077,  'SS high-similarity pair count differs - STOP'
print('\nhigh-similarity pair counts match colab36')

## 4. The two methods that tie

Both are computed from the strings alone.

* **Length ratio** — $\min(|a|,|b|)/\max(|a|,|b|)$.
* **Dice over 3-grams** — $2|A \cap B| / (|A| + |B|)$ on the *sets* of 3-grams, matching
  Section 3.5.2. Implemented as a sparse binary incidence matrix so the intersections come out of
  one matrix product.

Scores are only ever computed for **query rows**, not for the whole collection against itself —
that is what keeps this cheap after the scan above.

In [ ]:
def gram_matrix(seqs, k=3):
    vocab, rows, cols = {}, [], []
    for i, s in enumerate(seqs):
        gs = {s[j:j+k] for j in range(len(s) - k + 1)}
        for g in gs:
            c = vocab.setdefault(g, len(vocab))
            rows.append(i); cols.append(c)
    M = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                          shape=(len(seqs), len(vocab)))
    return M, np.asarray(M.sum(axis=1)).ravel()

def scores_for_queries(seqs, qidx, method, block=256):
    '''Returns (n_queries, N) score matrix for the given method.'''
    N = len(seqs); lens = np.array([len(s) for s in seqs], dtype=np.float64)
    out = np.empty((len(qidx), N), dtype=np.float32)
    if method == 'length':
        for b0 in range(0, len(qidx), block):
            b1 = min(b0 + block, len(qidx))
            ql = lens[qidx[b0:b1]][:, None]
            out[b0:b1] = (np.minimum(ql, lens[None, :]) /
                          np.maximum(ql, lens[None, :])).astype(np.float32)
    elif method == 'dice':
        M, sz = gram_matrix(seqs)
        for b0 in range(0, len(qidx), block):
            b1 = min(b0 + block, len(qidx))
            inter = np.asarray((M[qidx[b0:b1]] @ M.T).todense(), dtype=np.float32)
            den = (sz[qidx[b0:b1]][:, None] + sz[None, :]).astype(np.float32)
            den[den == 0] = 1.0
            out[b0:b1] = 2.0 * inter / den
    else:
        raise ValueError(method)
    return out

print('scorers defined')

## 5. Where the ties actually are

Before measuring the spread, measure the thing that causes it: at the rank-10 boundary, how many
candidates share the cut-off score? A query whose boundary block has size 1 has no ambiguity at
all; a query whose block has size 4,000 is almost entirely undetermined.

This table is worth having in its own right — it is the evidence for the claim in Section 3.6.5
that ties are systematic for these two methods rather than incidental.

In [ ]:
def prep_queries(seqs, rel, method):
    '''Per query: the fixed part of the top-10 and the tied block straddling the cut.'''
    qidx = np.array([i for i, x in enumerate(rel) if len(x) > 0], dtype=np.int64)
    S = scores_for_queries(seqs, qidx, method)
    prepped, diag = [], []
    for row, gi in enumerate(qidx):
        sc = S[row].astype(np.float64).copy()
        sc[gi] = -np.inf                                  # a sequence is not its own candidate
        relset = np.zeros(len(seqs), dtype=bool); relset[rel[gi]] = True
        kth = -np.partition(-sc, K_AT - 1)[K_AT - 1]      # the 10th largest score
        above = np.where(sc > kth)[0]
        tied  = np.where(sc == kth)[0]
        prepped.append(dict(
            R=int(len(rel[gi])),
            above_scores=sc[above].copy(),
            above_rel=relset[above].copy(),
            n_tie=int(len(tied)),
            r_tie=int(relset[tied].sum())))
        diag.append(len(tied))
    return qidx, prepped, np.array(diag)

TIEDIAG = {}
PREP = {}
for method in ['length', 'dice']:
    for r in ORDER:
        t0 = time.time()
        qidx, prepped, blocks = prep_queries(COLLECTION[r], REL[r], method)
        PREP[(method, r)] = prepped
        TIEDIAG[(method, r)] = dict(
            method=method, dataset=r, queries=len(qidx),
            frac_with_tie=float((blocks > 1).mean()),
            median_block=float(np.median(blocks)),
            max_block=int(blocks.max()),
            mean_block=float(blocks.mean()))
        print(f'  {method:<7} {r:<6} queries={len(qidx):>6,}  '
              f'ties at the cut in {100*(blocks>1).mean():5.1f}% of queries  '
              f'median block {np.median(blocks):>7.0f}  max {blocks.max():>6,}  '
              f'({time.time()-t0:.0f}s)')

TIES = pd.DataFrame(TIEDIAG.values())
TIES

## 6. MAP@10 under random tie-breaking

For one ordering: the candidates scoring strictly above the cut are always in the top ten, shuffled
within groups of equal score; the remaining slots are filled by drawing uniformly without
replacement from the tied block. How many of those drawn are relevant follows the hypergeometric
distribution, which is exactly what drawing without replacement means — so the draw is done
directly rather than by materialising and shuffling a block of several thousand candidates.

$$\mathrm{AP@10}(q)=\frac{1}{\min(R_q,10)}\sum_{k=1}^{10}\mathrm{rel}_q(k)\,P@k(q)$$

In [ ]:
def ap_at_10(rel_seq, R):
    '''rel_seq: 0/1 array of length K_AT in retrieved order.'''
    csum = np.cumsum(rel_seq)
    prec = csum / np.arange(1, K_AT + 1)
    return float((rel_seq * prec).sum() / min(R, K_AT))

def one_ordering(prepped, rng):
    aps = np.empty(len(prepped))
    for j, q in enumerate(prepped):
        asc, arel = q['above_scores'], q['above_rel']
        n_above = len(asc)
        if n_above:
            # random tie-break WITHIN equal-score groups of the above-cut part
            key = rng.random(n_above)
            order = np.lexsort((key, -asc))
            head = arel[order].astype(np.int8)
        else:
            head = np.zeros(0, dtype=np.int8)
        m = K_AT - n_above
        if m > 0 and q['n_tie'] > 0:
            take = min(m, q['n_tie'])
            h = rng.hypergeometric(q['r_tie'], q['n_tie'] - q['r_tie'], take)
            tail = np.zeros(take, dtype=np.int8)
            if h:
                tail[rng.choice(take, size=h, replace=False)] = 1
        else:
            tail = np.zeros(0, dtype=np.int8)
        seq = np.concatenate([head, tail])
        if len(seq) < K_AT:
            seq = np.concatenate([seq, np.zeros(K_AT - len(seq), dtype=np.int8)])
        aps[j] = ap_at_10(seq[:K_AT], q['R'])
    return float(aps.mean())

print('MAP@10 machinery defined')

In [ ]:
# B_MAX independent orderings per method/dataset. Each ordering yields one MAP@10 value.
DRAWS = {}
for (method, r), prepped in PREP.items():
    rng = np.random.default_rng(TIE_SEED)
    t0 = time.time()
    vals = np.array([one_ordering(prepped, rng) for _ in range(B_MAX)])
    DRAWS[(method, r)] = vals
    print(f'  {method:<7} {r:<6} MAP@10 over {B_MAX:,} orderings: '
          f'mean {vals.mean():.4f}  sd {vals.std(ddof=1):.5f}  '
          f'[{vals.min():.4f}, {vals.max():.4f}]   ({time.time()-t0:.0f}s)')

## 7. The answer

`sd_single_ordering` is how much MAP@10 moves if you break ties once, at random.
`se_at_200` is how much the **reported** number moves, and it is the one that matters:
it is $\sigma/\sqrt{200}$.

Compare `se_at_200` against the precision the thesis reports MAP@10 to. If the thesis quotes three
decimals and the standard error sits well below $0.0005$, then 200 orderings were already more than
enough and any other value in the same range would have produced the same table.

In [ ]:
rows = []
for (method, r), vals in DRAWS.items():
    sd = float(vals.std(ddof=1))
    rows.append(dict(method=method, dataset=r,
                     queries=TIEDIAG[(method, r)]['queries'],
                     map10_mean=round(float(vals.mean()), 5),
                     sd_single_ordering=round(sd, 6),
                     se_at_200=round(sd / np.sqrt(B_REPORTED), 6),
                     spread_1000=round(float(vals.max() - vals.min()), 6),
                     frac_queries_with_tie=round(TIEDIAG[(method, r)]['frac_with_tie'], 4),
                     median_tied_block=TIEDIAG[(method, r)]['median_block']))
SPREAD = pd.DataFrame(rows).sort_values(['method', 'dataset']).reset_index(drop=True)
SPREAD

## 8. Appendix figure — the running estimate against the number of orderings

Two panels, one per method, four datasets each in the locked deck palette. The vertical line marks
$B = 200$. The band is $\pm 2\,\sigma/\sqrt{B}$ around the $B = 1{,}000$ mean — the interval the
estimate is expected to sit in after $B$ orderings.

What to look for: the curve should be visually flat long before the 200 line, and the band should
be narrower than the precision the thesis reports.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
Bs = np.arange(1, B_MAX + 1)

for ax, method in zip(axes, ['length', 'dice']):
    for r in ORDER:
        vals = DRAWS[(method, r)]
        running = np.cumsum(vals) / Bs
        sd = vals.std(ddof=1)
        ax.plot(Bs, running, color=COLOUR[r], lw=1.4, label=r)
        ax.fill_between(Bs, vals.mean() - 2*sd/np.sqrt(Bs), vals.mean() + 2*sd/np.sqrt(Bs),
                        color=COLOUR[r], alpha=0.10, linewidth=0)
    ax.axvline(B_REPORTED, color='0.35', ls='--', lw=1.0)
    ax.annotate(f'B = {B_REPORTED}', xy=(B_REPORTED, ax.get_ylim()[1]),
                xytext=(4, -12), textcoords='offset points', fontsize=8, color='0.35')
    ax.set_xscale('log')
    ax.set_xlabel('number of random tie orderings $B$')
    ax.set_title({'length': 'Length ratio', 'dice': 'Dice over 3-grams'}[method], fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('running MAP@10 estimate')
axes[1].legend(frameon=False, fontsize=9, loc='best')
fig.tight_layout()
fig.savefig('colab39_tie_convergence.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Save

Downloads `colab39_tie_breaking.json`, the two CSVs and the PNG, and also writes them to Drive so
they survive the session. **The JSON is the receipt — download it before the runtime disconnects.**

In [ ]:
summary = dict(
    config=dict(B_max=B_MAX, B_reported=B_REPORTED, seed=TIE_SEED, k=K_AT, high=HIGH),
    collections={r: len(COLLECTION[r]) for r in ORDER},
    high_similarity_pairs=HIGH_PAIRS,
    eligible_queries={r: int(sum(1 for x in REL[r] if len(x) > 0)) for r in ORDER},
    tie_diagnostics=[TIEDIAG[k] for k in TIEDIAG],
    spread=SPREAD.to_dict('records'))

with open('colab39_tie_breaking.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
json.loads(open('colab39_tie_breaking.json').read())          # validate it parses
SPREAD.to_csv('colab39_tie_spread.csv', index=False)
TIES.to_csv('colab39_tie_diagnostics.csv', index=False)
print(json.dumps(summary, indent=2)[:2500])

try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = '/content/drive/MyDrive/thesis_colab_outputs'
    os.makedirs(dest, exist_ok=True)
    for f in ['colab39_tie_breaking.json', 'colab39_tie_spread.csv',
              'colab39_tie_diagnostics.csv', 'colab39_tie_convergence.png']:
        shutil.copy2(f, dest); print('saved to Drive:', f)
except Exception as e:
    print('Drive not mounted:', e)

from google.colab import files
for f in ['colab39_tie_breaking.json', 'colab39_tie_spread.csv',
          'colab39_tie_diagnostics.csv', 'colab39_tie_convergence.png']:
    files.download(f)

## 10. What this does and does not settle

**Settles:** whether the reported MAP@10 depends on the choice of 200. If `se_at_200` is far below
the reported precision, it does not, and that is a measurement rather than a preference — the same
shape of answer that the decile sweep gives for 200,000 and 400.

**Does not settle:** where the number 200 originally came from. Nothing in this notebook recovers
that, and the defence does not need it.

⚠ **The MAP@10 means printed here are this notebook's own estimates.** They should agree with the
recorded length-ratio and Dice numbers to within `se_at_200`. If they do not, that is a finding
about provenance, not about ties — check it before quoting anything from here in Chapter 4.